# 0. Setup & Installation
Run this cell first. Restart runtime if prompted.

In [ ]:
!pip install -U transformers peft bitsandbytes accelerate datasets scikit-learn --quiet

# 1. Authentication

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_1 = user_secrets.get_secret("HF_TOKEN")

login(token=secret_value_1)

# 2. Configuration

In [ ]:
from dataclasses import dataclass, field
from typing import List


@dataclass
class ExperimentConfig:
    # ---- Model ----
    model_name: str = "Qwen/Qwen3-4B"
    max_seq_length: int = 2048

    # ---- LoRA ----
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ])

    # ---- Training ----
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.1

    # ---- Quantization ----
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = "nf4"
    use_double_quant: bool = True

    # ---- BCE ----
    bce_weight: float = 1.0

    # ---- Prediction marker (plain text, NOT a special token) ----
    prediction_marker: str = "<|prediction|>"

    # ---- Paths ----
    output_dir: str = "./outputs"


cfg = ExperimentConfig()
print(f"Config: {cfg}")

# 3. Load Training Data

In [ ]:
import pandas as pd
from datasets import Dataset

train_data = pd.read_csv("/kaggle/input/notebooks/zygong1994/finetuning-experiment-0326-etl/train.csv")
eval_data = pd.read_csv("/kaggle/input/notebooks/zygong1994/finetuning-experiment-0326-etl/test.csv")

train_dataset = Dataset.from_pandas(train_data[["text"]].iloc[:500])
eval_dataset = Dataset.from_pandas(eval_data[["text"]])

print(f"Train: {len(train_dataset)} samples")
print(f"Eval:  {len(eval_dataset)} samples")

# 4. Load Model + Tokenizer (QLoRA)
No special tokens are added. The prediction marker `<|prediction|>` is treated as
regular text and tokenized into multiple sub-tokens by the base tokenizer.
This means we do NOT need `resize_token_embeddings` or `modules_to_save`,
which keeps the trainable parameter count small.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ---- Quantization config ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.load_in_4bit,
    bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=cfg.use_double_quant,
)

# ---- Load tokenizer (no special tokens added) ----
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ---- Encode the prediction marker as a multi-token sequence ----
prediction_marker_ids = tokenizer.encode(
    cfg.prediction_marker, add_special_tokens=False
)
print(f"Prediction marker '{cfg.prediction_marker}' -> token IDs: {prediction_marker_ids}")
print(f"  Decoded back: {[tokenizer.decode([t]) for t in prediction_marker_ids]}")

# ---- Get label token IDs for BCE loss ----
label_0_token_id = tokenizer.encode("0", add_special_tokens=False)[0]
label_1_token_id = tokenizer.encode("1", add_special_tokens=False)[0]

print(f"Token IDs -> '0': {label_0_token_id}, '1': {label_1_token_id}")
assert label_0_token_id != tokenizer.unk_token_id, "'0' mapped to UNK!"
assert label_1_token_id != tokenizer.unk_token_id, "'1' mapped to UNK!"

# ---- Load model (no embedding resize needed) ----
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",  
    torch_dtype=torch.bfloat16
)

print(f"Model loaded. Vocab size: {model.config.vocab_size}")

# 5. Apply LoRA
Without `modules_to_save`, only the LoRA adapter weights are trainable.
The full `embed_tokens` and `lm_head` layers stay frozen.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    target_modules=cfg.lora_target_modules,
    lora_dropout=cfg.lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

# 6. Tokenize with Label Masking
Everything up to and including `<|prediction|>` is masked (`labels=-100`).
Only the label token (`0`/`1`) and closing marker are trained on.
This focuses all gradient signal on the classification task.

In [ ]:
from transformers import DataCollatorForSeq2Seq


def find_subseq(seq, subseq):
    """Find starting index of subseq in a list. Returns -1 if not found."""
    n = len(subseq)
    for i in range(len(seq) - n + 1):
        if seq[i:i + n] == subseq:
            return i
    return -1


def tokenize_and_mask(examples):
    tokenized = tokenizer(
        examples["text"], truncation=True, max_length=cfg.max_seq_length
    )
    all_labels = []
    for input_ids in tokenized["input_ids"]:
        labels = input_ids.copy()
        idx = find_subseq(input_ids, prediction_marker_ids)
        if idx != -1:
            # Mask everything up to and including <|prediction|>
            mask_end = idx + len(prediction_marker_ids)
            labels[:mask_end] = [-100] * mask_end
        all_labels.append(labels)
    tokenized["labels"] = all_labels
    return tokenized


train_tokenized = train_dataset.map(tokenize_and_mask, batched=True, remove_columns=["text"])
eval_tokenized = eval_dataset.map(tokenize_and_mask, batched=True, remove_columns=["text"])

data_collator = DataCollatorForSeq2Seq(tokenizer, padding=True, pad_to_multiple_of=8)

# ---- Verify masking ----
sample = train_tokenized[0]
masked_count = sum(1 for l in sample["labels"] if l == -100)
total = len(sample["labels"])
unmasked_ids = [t for t in sample["labels"] if t != -100]
print(f"Sample: {total} tokens total, {masked_count} masked, {total - masked_count} trained on")
print(f"Unmasked tokens: '{tokenizer.decode(unmasked_ids)}'")

# 7. BCE + Masked LM Trainer
Uses standard `Trainer` (not SFTTrainer) since we pre-tokenized with custom labels.
LM loss only fires on unmasked positions (after `<|prediction|>`).
BCE loss targets the prediction marker position as before.

In [ ]:
import torch.nn.functional as F
from torch.nn import BCEWithLogitsLoss
from transformers import Trainer, TrainingArguments


class BCETrainer(Trainer):
    """
    Extends Trainer to add BCE loss on the <|prediction|> marker position.

    LM loss is computed only on unmasked label positions (after the marker).
    BCE loss uses the logit difference between '1' and '0' tokens at the
    last marker position to compute binary cross-entropy against ground truth.
    """

    def __init__(self, *args, bce_weight=1.0, prediction_marker_ids=None,
                 label_0_token_id=None, label_1_token_id=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.bce_weight = bce_weight
        self.prediction_marker_ids = prediction_marker_ids
        self.label_0_token_id = label_0_token_id
        self.label_1_token_id = label_1_token_id
        self.bce_loss_fn = BCEWithLogitsLoss()

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # ---- Masked LM loss (only on unmasked positions) ----
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        lm_loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=-100,
        )

        # ---- BCE loss on prediction marker position ----
        input_ids = inputs["input_ids"]
        batch_size = input_ids.size(0)
        marker_ids = self.prediction_marker_ids

        bce_losses = []
        for b in range(batch_size):
            seq = input_ids[b].tolist()
            start_idx = find_subseq(seq, marker_ids)

            if start_idx == -1:
                continue

            pred_pos = start_idx + len(marker_ids) - 1

            if pred_pos + 1 < input_ids.size(1):
                logit_0 = logits[b, pred_pos, self.label_0_token_id]
                logit_1 = logits[b, pred_pos, self.label_1_token_id]

                actual_next = input_ids[b, pred_pos + 1].item()
                target = torch.tensor(
                    1.0 if actual_next == self.label_1_token_id else 0.0,
                    device=logits.device,
                )
                bce_logit = logit_1 - logit_0
                bce_losses.append(self.bce_loss_fn(bce_logit, target))

        if bce_losses:
            bce_loss_mean = torch.stack(bce_losses).mean()
            total_loss = lm_loss + self.bce_weight * bce_loss_mean
        else:
            total_loss = lm_loss
            bce_loss_mean = torch.tensor(0.0, device=lm_loss.device)

        if self.state.global_step % self.args.logging_steps == 0:
            self.log({
                "lm_loss": lm_loss.detach().item(),
                "bce_loss": bce_loss_mean.detach().item(),
                "total_loss": total_loss.detach().item(),
            })

        return (total_loss, outputs) if return_outputs else total_loss

# 8. Training

In [ ]:
training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    warmup_ratio=cfg.warmup_ratio,
    lr_scheduler_type="cosine",
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="no",
    seed=42,
    report_to="none",
)

trainer = BCETrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator,
    bce_weight=cfg.bce_weight,
    prediction_marker_ids=prediction_marker_ids,
    label_0_token_id=label_0_token_id,
    label_1_token_id=label_1_token_id,
)

print("Starting training...")
train_result = trainer.train()
print(f"\nTraining complete!")
print(f"Final train loss: {train_result.training_loss:.4f}")

# 9. Save LoRA Adapter

In [ ]:
import os, json

adapter_path = f"{cfg.output_dir}/final_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

for f in os.listdir(adapter_path):
    size_mb = os.path.getsize(os.path.join(adapter_path, f)) / 1024 / 1024
    print(f"  {f}: {size_mb:.2f} MB")

with open(os.path.join(adapter_path, "experiment_config.json"), "w") as fp:
    json.dump(vars(cfg), fp, indent=2)